# WAXAL — Expand KenLM text corpora on Colab (GPU-free)

Harvests transcripts (TEXT ONLY, no audio decode) from WAXAL + external open datasets,
builds bigger per-language KenLMs into `data/lm_expanded/`, and pushes them to HF.

**Does not touch the working pipeline.** Runs scripts with plain `python` (not `uv`/`make`),
and writes to `data/lm_expanded/` so the proven `data/lm/` LMs are untouched.

Use a **CPU / High-RAM runtime** for cells 1-5 (don't burn A100 units on text work).
The optional GPU sweep at the bottom is the only part that wants an A100.

## 1. Clone repo + install text-harvest deps

In [ ]:
import os
if not os.path.isdir('waxal-zindi-challenge'):
    !git clone https://github.com/yigagilbert/waxal-zindi-challenge.git
%cd waxal-zindi-challenge
!git pull
# text harvest needs only `datasets` (+ huggingface_hub for gated access & push)
!pip install -q 'datasets>=3.0.0' huggingface_hub

## 2. Hugging Face login
Needed for the gated sources (Afrivoice, your Yogera repo) and to push results.
**Accept each gated dataset's terms in the browser first**, then paste a write token.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Build the KenLM CLI (`lmplz`, `build_binary`) — ~3 min
Only needed to build the `.binary` LMs. Skip if you only want the text corpora.

In [ ]:
import shutil
if shutil.which('lmplz') is None:
    !apt-get -qq install -y build-essential cmake libboost-all-dev libeigen3-dev zlib1g-dev libbz2-dev liblzma-dev >/dev/null
    !git clone -q https://github.com/kpu/kenlm.git /content/kenlm
    !mkdir -p /content/kenlm/build && cd /content/kenlm/build && cmake .. -DCMAKE_BUILD_TYPE=Release >/dev/null && make -j2 >/dev/null
    os.environ['PATH'] = '/content/kenlm/build/bin:' + os.environ['PATH']
print('lmplz:', shutil.which('lmplz'))

## 4. Harvest text + build expanded LMs
`--waxal-from-hf` pulls the in-domain WAXAL train text from HF (no local CSV needed).
Capped per source + WAXAL doubled so Common Voice doesn't drown out in-domain Luganda.
Sources that fail to load (wrong gated schema) are skipped with a warning, not fatal.

In [ ]:
!python scripts/collect_lm_text.py \
  --waxal-from-hf --waxal-repeat 2 \
  --max-lines-per-source 60000 \
  --output-dir data/lm_expanded --order 5 --overwrite

In [ ]:
import json
rep = json.load(open('data/lm_expanded/lm_text_provenance.json'))
print('corpora lines:', rep['corpora'])
print()
for p in rep['provenance']:
    print(p.get('source') or p.get('dataset'), p.get('config',''), '->', p.get('lines'), '|', p.get('license'))

**Check the provenance above:** every source should show a nonzero line count. If
`common-voice-sample-packed-lug` shows 0, its 'packed' schema isn't a standard
`audio`+`sentence` layout — report the real schema and we'll add a loader. If Afrivoice
shows 0, its config/column names differ from the guess — same fix.

## 5. Push the expanded LMs to HF (so any GPU box can pull them)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo = 'yigagilbert/waxal-xlsr300m-champion'  # same private model repo as the champion + working LMs
api.create_repo(repo, repo_type='model', private=True, exist_ok=True)
api.upload_folder(folder_path='data/lm_expanded', path_in_repo='lm_expanded', repo_id=repo, repo_type='model')
print('pushed lm_expanded ->', repo)

## 6. (Optional, needs A100) Re-sweep with the expanded LMs
Switch to a **GPU runtime** for this. Compares tuned decode with the expanded LMs
vs the current ones. The headline test: does **sna** flip to `best_beats_greedy: true`?

In [ ]:
# needs the champion model + the prepared generalization-mix validation on disk.
# Pull the champion checkpoint from HF, and (re)generate the validation split, then:
!pip install -q 'transformers>=4.46,<5' torch pyctcdecode

from huggingface_hub import snapshot_download
!snapshot_download('yigagilbert/waxal-xlsr300m-champion', local_dir='champion')
!python scripts/sweep_kenlm_decode_params.py --checkpoint champion/checkpoint-24000 \
  --dataset-dir data/processed_generalization_mix --kenlm-dir data/lm_expanded --order 5 \
  --output outputs/analysis/kenlm_alpha_beta_sweep_expanded.json